# Notebook 03 - feature engineering
Turns the clean lap table into a model-ready table. Reads `data/clean_laps.csv.gz`, writes `data/model_table.csv.gz`. All the logic lives in `f1lap.features`.

**Leakage rule for every feature:** it must be knowable *before the lap is driven*. Nothing from the same lap, a later lap, or the race result.

In [1]:
import os, sys
# Run from the repo root so data/, figures/ and results/ resolve, and make src/ importable.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.path.abspath('src'))

import pandas as pd
from f1lap.features import add_features, encode_features, NUM_FEATURES

df = add_features(pd.read_csv('data/clean_laps.csv.gz'))
df.shape

(90488, 25)

### 1. Features
| Feature | Type | Why |
|---|---|---|
| `tyre_age` | numeric | laps on this set of tyres, the main degradation driver |
| `fuel_proxy` | numeric | laps remaining in the race, standing in for fuel load (cars burn ~1.5-2 kg per lap) |
| `stint` | numeric | which stint this is, so the model can tell a first stint from a final sprint |
| `position` | numeric | running position, which captures traffic and (indirectly) car pace |
| `air_temp_c`, `track_temp_c` | numeric | grip and degradation depend on temperature |
| `compound` | one-hot | soft / medium / hard have different pace and wear |
| `team_lineage` | one-hot | which car it is, with rebrands merged (AlphaTauri/RB -> Racing Bulls, Alfa Romeo/Kick Sauber/Audi -> Sauber/Audi) |

Circuits are matched by circuit, not event name, because names lie: in 2026 the "Spanish GP" moved to a new Madrid circuit, Barcelona hosts the "Barcelona GP", and the "Bahrain GP" was held at Sepang.

### 2. Entities the model has never seen

In [2]:
print('Teams in 2026 not seen in 2022-2025:', sorted(df.loc[df.unseen_team, 'team'].unique()))
train_drivers = set(df.loc[df.season < 2026, 'driver'])
print('2026 drivers with no 2022-2025 race laps:', sorted(set(df.loc[df.season == 2026, 'driver']) - train_drivers))
print('2026 circuits not raced dry in 2022-2025:',
      sorted(set(df.loc[df.season == 2026, 'circuit']) - set(df.loc[df.season < 2026, 'circuit'])))

Teams in 2026 not seen in 2022-2025: ['Cadillac']
2026 drivers with no 2022-2025 race laps: ['CAM', 'HIR', 'LIN']
2026 circuits not raced dry in 2022-2025: ['Madrid']


These are the hardest cases for any model trained on the past. Cadillac gets its own one-hot column that is all zeros in training, so until the model sees 2026 data it treats Cadillac like an "average" team. We report their errors separately in notebook 05.

### 3. Encoding

In [3]:
X = encode_features(df)
print('Feature matrix:', X.shape, '| missing values:', int(X.isna().sum().sum()))
df[NUM_FEATURES].describe().round(2)

Feature matrix: (90488, 20) | missing values: 0


,tyre_age,fuel_proxy,stint,position,air_temp_c,track_temp_c
count,90488.00,90488.00,90488.00,90488.00,90488.00,90488.00
mean,15.23,29.88,2.01,9.77,24.76,37.31
std,9.99,18.00,0.89,5.45,4.97,9.06
min,1.00,0.00,1.00,1.00,11.60,13.30
25%,8.00,14.00,1.00,5.00,21.10,31.10
50%,13.00,30.00,2.00,10.00,25.40,37.00
75%,21.00,44.00,3.00,14.00,28.80,44.10
max,78.00,76.00,8.00,22.00,37.20,57.20


**Encoding choices.** Compound and team are one-hot encoded (3 and 11 columns). We don't scale numeric features: XGBoost splits on thresholds so scale doesn't matter, and the physics baseline is an unregularised linear regression, whose fit is identical with or without scaling. The matrix has zero missing values, so no imputation is needed.

### 4. Save the model table

In [4]:
df.to_csv('data/model_table.csv.gz', index=False)
print('Saved data/model_table.csv.gz', df.shape)

Saved data/model_table.csv.gz (90488, 25)
